# 25.3 — Differentiable programming

**Lesson tagline.** Differentiable programming writes a model as ordinary code, then chooses smooth pieces so gradients can tune it.

A hard program branch can be correct but untrainable. This notebook turns a threshold program into a differentiable sigmoid program and trains its parameter across a difficulty ladder.

Save a copy to Drive to edit.

In [ ]:

import math
import itertools
import random
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

SEED = 25
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))


def binary_cross_entropy(p, y):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1.0 - 1e-12)
    y = np.asarray(y, dtype=float)
    terms = -y * np.log(p) - (1.0 - y) * np.log(1.0 - p)
    return float(np.mean(terms))



def soft_threshold_program(x, t, k):
    return sigmoid(k * (np.asarray(x, dtype=float) - t))


def threshold_loss(x, y, t, k):
    return binary_cross_entropy(soft_threshold_program(x, t, k), y)


def finite_difference_grad(x, y, t, k, eps=1e-4):
    plus = threshold_loss(x, y, t + eps, k)
    minus = threshold_loss(x, y, t - eps, k)
    return (plus - minus) / (2.0 * eps)


def train_threshold(x, y, t0=1.0, k=4.0, lr=0.2, steps=30):
    t = float(t0)
    history = []
    for step in range(steps):
        loss = threshold_loss(x, y, t, k)
        grad = finite_difference_grad(x, y, t, k)
        history.append((step, t, loss, grad))
        t = t - lr * grad
    history.append((steps, t, threshold_loss(x, y, t, k), finite_difference_grad(x, y, t, k)))
    return t, history


def make_diffprog_ladder(seed=25):
    rng = np.random.default_rng(seed)
    rungs = []
    x1 = np.array([0.0, 1.0, 2.0, 3.0])
    y1 = np.array([0, 0, 1, 1])
    rungs.append({'name': 'D1 lesson threshold toy', 'x': x1, 'y': y1, 'k': 4.0, 't0': 1.0})
    x2 = np.linspace(-2.0, 3.0, 40)
    y2 = (x2 > 0.7).astype(float)
    rungs.append({'name': 'D2 clean 1D threshold', 'x': x2, 'y': y2, 'k': 5.0, 't0': 0.0})
    x3 = np.linspace(-2.0, 3.0, 80)
    logits = x3 - 0.8 + 0.45 * rng.normal(size=x3.shape)
    y3 = (logits > 0.0).astype(float)
    rungs.append({'name': 'D3 noisy overlapping threshold', 'x': x3, 'y': y3, 'k': 4.0, 't0': 0.0})
    x4 = np.linspace(-3.0, 3.0, 120)
    risk = 0.9 * x4 + 0.35 * np.sin(2.0 * x4)
    y4 = (risk > 0.4).astype(float)
    rungs.append({'name': 'D4 real-ish one-feature risk score', 'x': x4, 'y': y4, 'k': 4.0, 't0': -0.5})
    x5 = np.linspace(-3.0, 3.0, 160)
    aux = np.cos(2.5 * x5)
    y5 = ((x5 + 0.55 * aux) > 0.45).astype(float)
    rungs.append({'name': 'D5 constrained two-feature sharpness stress', 'x': x5, 'aux': aux, 'y': y5, 'k': 18.0, 't0': -1.5})
    return rungs


## The concept, built once (D1)

The relaxed threshold program is $\hat y_i(t)=\sigma(k(x_i-t))$, trained by minimizing BCE $L(t)=rac1m\sum_i BCE(\hat y_i(t),y_i)$.

In [ ]:

x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([0, 0, 1, 1])
predictions = soft_threshold_program(x, t=1.5, k=4.0)
loss_at_center = threshold_loss(x, y, t=1.5, k=4.0)
print(predictions)
print(loss_at_center)
assert np.allclose(predictions, np.array([0.0024726231566347743, 0.11920292202211755, 0.8807970779778823, 0.9975273768433653]))
assert abs(loss_at_center - 0.06470184809035148) < 1e-15
assert abs(threshold_loss(x, y, 1.0, 4.0) - 0.41003759580145864) < 1e-15
assert abs(threshold_loss(x, y, 2.0, 4.0) - 0.41003759580145864) < 1e-15


A finite-difference gradient makes the program parameter trainable. From $t=1.0$, the lesson gradient is $-0.49966464986062054$ and 30 updates reach $tpprox1.4984574318643302$.

In [ ]:

grad = finite_difference_grad(x, y, t=1.0, k=4.0, eps=1e-4)
learned_t, history = train_threshold(x, y, t0=1.0, k=4.0, lr=0.2, steps=30)
print('gradient at 1.0:', grad)
print('learned threshold:', learned_t)
assert abs(grad - (-0.49966464986062054)) < 1e-12
assert abs(learned_t - 1.4984574318643302) < 1e-12


## The dataset ladder

The F4 ladder makes the objective harder: soft-threshold toy, more points, noisy labels, real-ish risk score, then a constrained sharpness-prone two-feature case.

In [ ]:

rungs = make_diffprog_ladder()
for i, rung in enumerate(rungs, start=1):
    print(f"D{i}: {rung['name']} n={len(rung['x'])} positive_rate={np.mean(rung['y']):.3f} k={rung['k']}")
    print('sample x,y:', list(zip(rung['x'][:5], rung['y'][:5])))


## Run the SAME method across D1–D5

In [ ]:

results = []
for i, rung in enumerate(rungs, start=1):
    train_x = rung['x'] + 0.35 * rung.get('aux', np.zeros_like(rung['x']))
    learned_t, history = train_threshold(train_x, rung['y'], t0=rung['t0'], k=rung['k'], lr=0.1, steps=80)
    final_loss = threshold_loss(train_x, rung['y'], learned_t, rung['k'])
    results.append({'D': i, 'name': rung['name'], 't': learned_t, 'loss': final_loss, 'history': history})
print('rung | final_bce | learned_t')
for row in results:
    print(f"D{row['D']} | {row['loss']:.4f} | {row['t']:.4f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, rung in enumerate(rungs):
    train_x = rung['x'] + 0.35 * rung.get('aux', np.zeros_like(rung['x']))
    row = results[col]
    order = np.argsort(train_x)
    axes[0, col].scatter(train_x, rung['y'], s=15, alpha=0.6)
    axes[0, col].plot(train_x[order], soft_threshold_program(train_x[order], row['t'], rung['k']), color='black')
    axes[0, col].set_title(f"D{col + 1} sigmoid fit")
axes[1, 0].plot([h[0] for h in results[0]['history']], [h[2] for h in results[0]['history']])
axes[1, 0].set_title('D1 loss curve')
axes[1, 1].plot([row['D'] for row in results], [row['loss'] for row in results], marker='o', label='final BCE')
axes[1, 1].plot([row['D'] for row in results], [row['t'] for row in results], marker='s', label='learned t')
axes[1, 1].set_title('metric vs complexity')
axes[1, 1].legend()
for ax in axes[1, 2:]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## Pitfall on the hardest rung

If the relaxation is too sharp too early, gradients saturate and the program behaves like a hard branch. Annealing $k$ keeps gradients alive.

In [ ]:

d5 = rungs[-1]
train_x = d5['x'] + 0.35 * d5['aux']
sharp_t, sharp_history = train_threshold(train_x, d5['y'], t0=d5['t0'], k=60.0, lr=0.1, steps=30)
print('sharp final t and loss:', sharp_t, threshold_loss(train_x, d5['y'], sharp_t, 60.0))
print('sharp gradient norms:', [abs(item[3]) for item in sharp_history[:5]])
t = d5['t0']
annealed = []
for k in [2.0, 4.0, 8.0, 12.0, 18.0]:
    t, local_history = train_threshold(train_x, d5['y'], t0=t, k=k, lr=0.08, steps=20)
    annealed.append((k, t, threshold_loss(train_x, d5['y'], t, k), abs(local_history[-1][3])))
print('annealed stages:', annealed)


## Evaluate it + Practice

- **Metric.** Track binary cross-entropy loss, and compare it with a hard threshold chosen by grid search without gradients.
- **Cheap sanity check.** D1 must reproduce predictions and BCE from the lesson exactly.
- **Ablation.** replace sigmoid with a hard comparison and observe zero gradients on flat intervals.
- **Failure signals.** gradient norms collapse while loss remains high.

### Practice prompts


- Try a different learning rate schedule for D5.

- Add a second trainable parameter for sharpness k.

- Compare finite-difference and analytic gradients.